# Week 1-5 · evidence gate와 FastAPI 경계 조립하기

## 시나리오
작은 정책 검색 함수와 evidence gate를 FastAPI endpoint에 연결하고, 지원·미지원 질문을 `TestClient`로 비교합니다.

## 학습 목표
- 근거 유무로 답변 허용 여부를 결정한다.
- Pydantic request/response와 FastAPI endpoint를 직접 만든다.
- HTTP 성공 여부뿐 아니라 업무 상태와 citation을 검증한다.

## 직접 조립
완성된 `weekX.app` 함수를 가져오지 않습니다. 아래 코드에서 작은 fixture와 핵심 객체·함수·연결을 직접 만듭니다.

### 1단계 · 축소 RAG 함수

In [ ]:
# 실행 순서: 1단계 · 축소 RAG 함수에서 practice_retrieve, evidence_gate, PracticeRequest을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 1단계 · 축소 RAG 함수.
from pydantic import BaseModel, Field
from fastapi import FastAPI
from fastapi.testclient import TestClient

practice_policy = {"chunk_id": "leave-01", "text": "휴가는 시작일 3영업일 전에 신청합니다."}

# 질문에 맞는 정책만 반환해 retrieval의 최소 경계를 재현합니다.
def practice_retrieve(question: str) -> list[dict]:
    return [practice_policy] if "휴가" in question else []

# citation에 필요한 필드가 있는 근거만 답변 단계로 통과시킵니다.
def evidence_gate(documents: list[dict]) -> bool:
    return bool(documents and all(doc.get("chunk_id") and doc.get("text") for doc in documents))

# FastAPI가 endpoint 진입 시 질문 길이를 검증하는 request schema입니다.
class PracticeRequest(BaseModel):
    question: str = Field(min_length=3)

# 검증된 질문과 처리 상태를 호출자에게 돌려주는 최소 응답 계약입니다.
class PracticeResponse(BaseModel):
    status: str
    answer: str | None
    citations: list[str]

### 2단계 · endpoint 직접 연결

In [ ]:
# 실행 순서: 2단계 · endpoint 직접 연결에서 practice_query을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 2단계 · endpoint 직접 연결.
practice_api = FastAPI()

# request → retrieval → evidence gate → response 순서를 endpoint에 고정합니다.
@practice_api.post("/practice/query", response_model=PracticeResponse)
def practice_query(request: PracticeRequest) -> PracticeResponse:
    documents = practice_retrieve(request.question)
    if not evidence_gate(documents):
        return PracticeResponse(status="insufficient_evidence", answer=None, citations=[])
    return PracticeResponse(
        status="answered",
        answer=documents[0]["text"],
        citations=[documents[0]["chunk_id"]],
    )

practice_client = TestClient(practice_api)

### 3단계 · 지원·미지원 경로 테스트

In [ ]:
# 실행 순서: 3단계 · 지원·미지원 경로 테스트에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 3단계 · 지원·미지원 경로 테스트.
supported = practice_client.post("/practice/query", json={"question": "휴가는 언제 신청하나요?"})
unsupported = practice_client.post("/practice/query", json={"question": "와이파이 비밀번호는?"})
assert supported.status_code == 200
assert supported.json()["status"] == "answered" and supported.json()["citations"] == ["leave-01"]
assert unsupported.json() == {"status": "insufficient_evidence", "answer": None, "citations": []}
{"supported": supported.json(), "unsupported": unsupported.json()}

## 중간 결과
각 코드 셀의 출력에서 입력이 어떤 상태로 변했는지 확인합니다. 마지막 `assert`는 눈으로 본 결과를 실행 가능한 계약으로 고정합니다.

## 실패 경계
근거가 없으면 `insufficient_evidence`, `answer=None`, 빈 citation을 반환합니다. 실제 휴가 신청이나 승인은 수행하지 않습니다.

## 실제 app 연결
Week 1 app의 전체 흐름도 `retrieve → answer → grounding guard → API response`입니다. 여기서는 검색을 작은 함수로 줄여 API 경계와 fail-closed 계약에 집중했습니다.

### 확장 과제
fixture의 문장이나 임계값을 하나 바꾸고, 어느 중간 결과와 assertion이 달라지는지 기록하세요.

## 다음 Notebook 연결
다음 Week 2의 `01_structured_ticket_classification.ipynb`에서는 자유 형식 입력을 graph가 사용할 제한된 판단 값으로 변환합니다.